# 05 — Outil de triage et Model Registry

**Objectif** : exposer le JEM comme un outil qui ordonne un backlog de bugs non triés, publié dans le Model Registry MLflow.

**Fonctionnement**

- `Prioriser` regroupe le featurizer, le réseau et la distribution des énergies d'entraînement. Pour chaque bug : priorité prédite, confiance, probabilités, **score d'urgence**, percentile d'énergie et drapeau `needs_review` (énergie au-delà du 95e percentile de l'entraînement).
- Le modèle est sérialisé au format **pyfunc** MLflow (interface générique `predict(DataFrame) → DataFrame`), avec `pyproject.toml` et `uv.lock`.
- Le Model Registry versionne le modèle `jem-prioriser` ; l'alias **`champion`** désigne la version servie. `jem-predict` charge `models:/jem-prioriser@champion` : changer de modèle revient à déplacer l'alias.

In [1]:
import logging
import os
import warnings
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")  # chemins relatifs à la racine du projet (data/, mlflow.db)

warnings.filterwarnings("ignore")
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd

from jem_prioriser.plotstyle import CLASS_COLORS, HUMAN_BOT_COLORS, MODEL_COLORS, SERIES, use_style

logging.getLogger("mlflow").setLevel(logging.ERROR)
use_style()
pd.set_option("display.max_colwidth", 90)

## 1. Publication dans le Model Registry

In [2]:
from mlflow import MlflowClient
from jem_prioriser.train import EXPERIMENT, MODEL_NAME, TRACKING_URI

mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient()
jem_run = mlflow.search_runs(experiment_names=[EXPERIMENT], filter_string="tags.mlflow.runName = 'jem'",
                             order_by=["start_time DESC"]).iloc[0]
# idempotent : réutilise la version existante de ce run si elle est déjà enregistrée
existing = [v for v in client.search_model_versions(f"name = '{MODEL_NAME}'") if v.run_id == jem_run.run_id] \
    if client.search_registered_models(filter_string=f"name = '{MODEL_NAME}'") else []
version = existing[0] if existing else mlflow.register_model(f"runs:/{jem_run.run_id}/prioriser", MODEL_NAME)
client.set_registered_model_alias(MODEL_NAME, "champion", version.version)
print(f"{MODEL_NAME} version {version.version} -> alias 'champion'")

jem-prioriser version 1 -> alias 'champion'


Successfully registered model 'jem-prioriser'.
Created version '1' of model 'jem-prioriser'.


## 2. Tri du backlog Mozilla non trié

In [3]:
from jem_prioriser.dataset import load_untriaged

champion = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
ranked = champion.predict(load_untriaged())
print(f"{len(ranked)} bugs non triés ordonnés | signalés 'needs_review' : {ranked.needs_review.sum()} ({ranked.needs_review.mean():.1%})")
ranked[["rank", "summary", "predicted", "urgency", "confidence", "energy_percentile", "needs_review"]].head(10)

1849 bugs non triés ordonnés | signalés 'needs_review' : 99 (5.4%)


,rank,summary,predicted,urgency,confidence,energy_percentile,needs_review
0,1,"[Experiment] The ""PDF_ANNOTATIONS_HIGHLIGHT"" callout is not fully visible when triggered",P1,0.8848,0.811,6.7,False
1,2,Detect if a profile has Fx as default and then switched to Edge,P1,0.8841,0.737,29.3,False
2,3,browser/components/attribution/test/browser/browser_AttributionCode_telemetry.js will ...,P1,0.8801,0.740,45.9,False
3,4,[Experiment] Spotlight modal with Terms of Use information is no longer triggered afte...,P1,0.8700,0.795,15.4,False
4,5,[META] [QA] [QA-1809] Feature name Easy Setup Onboarding,P1,0.8623,0.710,21.6,False
5,6,"Crash [@ js::gc::GetOrCreateUniqueId] or Assertion failure: env, at vm/EnvironmentObje...",P1,0.8621,0.724,11.0,False
6,7,[META] [QA] [QA-1841] QA for FXE-182: Set to default for 1 week,P1,0.8600,0.698,66.5,False
7,8,[Experiment] The “Recommended Media Addons” Panel is not dismissed when navigating bet...,P1,0.8574,0.706,7.8,False
8,9,[META] [QA] [QA-2137] VPN Moments Pages October - [New & Existing],P1,0.8549,0.698,29.5,False
9,10,"Assertion failure: !hasDelayedMarking(), at gc/GC.cpp:3135",P1,0.8485,0.729,12.0,False


In [4]:
dist = ranked.predicted.value_counts().reindex(["P1", "P2", "P3", "Basse"]).to_frame("bugs")
dist["part"] = (dist.bugs / len(ranked)).round(3)
dist

,bugs,part
predicted,,
P1,202,0.109
P2,91,0.049
P3,1225,0.663
Basse,331,0.179


In [5]:
ranked.sort_values("energy", ascending=False)[["summary", "predicted", "confidence", "energy_percentile"]].head(8)

,summary,predicted,confidence,energy_percentile
1730,[meta] Consider accepting externally maintained patches,P3,0.621,100.0
1826,Certificate Manager | Making the searches a bit more efficient,Basse,0.548,100.0
1412,Consider rewriting createWindowsIcon within JavaScript,P3,0.805,99.9
1556,Intermittent /css/CSS2/bidi-005.xht | single tracking bug,P3,0.730,99.9
790,Gaps between border and content and background-image at some CSS scales / resolutions,P3,0.776,99.9
1359,Consider lowering the threshold for deeming swapchain to be slow for video palyback (W...,P3,0.747,99.9
1778,The tests of popover-focus*.html are failing,P3,0.537,99.9
1805,Intermittent TEST-UNEXPECTED-CRASH | netwerk/test/unit/test_XHR_redirects.js | single ...,P3,0.502,99.8


- En tête : expériences Nimbus, onboarding, QA de fonctionnalités et assertions du moteur JavaScript, cohérent avec le profil des P1 appris.
- 66 % de P3 prédits, proche de la part de P3 parmi les bugs triés.
- Les plus atypiques sont des tickets de réflexion (« Consider … », `[meta]`) et des échecs intermittents saisis à la main.

## 3. Nouveaux rapports

In [6]:
new_bugs = pd.DataFrame([
    {"summary": "Crash on startup after updating to Nightly", "description": "Firefox crashes immediately on launch since today's update. Crash report attached. Happens on every start.",
     "product": "Core", "component": "General", "type": "defect", "op_sys": "Windows 11"},
    {"summary": "Nimbus experiment enrollment fails for new users", "description": "The onboarding experiment does not enroll new profiles in Beta, blocking the rollout planned for this release.",
     "product": "Firefox", "component": "Messaging System", "type": "defect", "op_sys": "All"},
    {"summary": "Typo in bookmark folder context menu", "description": "The label says 'Delete foler' instead of 'Delete folder'.",
     "product": "Firefox", "component": "Bookmarks & History", "type": "defect", "op_sys": "Linux"},
    {"summary": "Add keyboard shortcut to duplicate a tab", "description": "It would be nice to have a shortcut to duplicate the current tab.",
     "product": "Firefox", "component": "Tabbed Browser", "type": "enhancement", "op_sys": "macOS"},
    {"summary": "Pizza party on Friday", "description": "Please bring your own drinks. Reminder: the parking lot is closed for cleaning.",
     "product": "Firefox", "component": "General", "type": "task", "op_sys": "Unspecified"},
])
champion.predict(new_bugs)[["rank", "summary", "predicted", "urgency", "confidence", "energy_percentile", "needs_review"]]

,rank,summary,predicted,urgency,confidence,energy_percentile,needs_review
0,1,Nimbus experiment enrollment fails for new users,P1,0.9030,0.773,20.3,False
1,2,Crash on startup after updating to Nightly,P3,0.4369,0.679,11.5,False
2,3,Pizza party on Friday,P3,0.4050,0.475,62.2,False
3,4,Typo in bookmark folder context menu,Basse,0.2685,0.604,42.4,False
4,5,Add keyboard shortcut to duplicate a tab,Basse,0.1968,0.512,81.6,False


- Le blocage d'une expérience Nimbus est classé P1 avec une bonne confiance : profil type des P1 Mozilla.
- Le crash au démarrage n'est classé que P3 : le modèle reproduit les habitudes de triage (beaucoup de crashs *Core :: General* finissent en P3 après analyse), au détriment de la gravité perçue.
- Le message hors sujet n'est pas signalé (62e percentile) : ses mots sont absents du vocabulaire TF-IDF, son vecteur ressemble à celui d'un bug très court. La détection est limitée à ce que la représentation capture ; des embeddings de phrases lèveraient cette limite.

## 4. Interface en ligne de commande

`jem-predict` trie le backlog de la base, ou tout CSV (`--input bugs.csv --output tries.csv`) contenant `summary`, `description`, `product`, `component`, `type`, `op_sys`.

In [7]:
import subprocess

out = subprocess.run(["uv", "run", "jem-predict", "--top", "8"], capture_output=True, text=True, encoding="utf-8", errors="replace")
print(out.stdout)

 rank                                                                summary predicted  urgency  confidence  needs_review
    1 [Experiment] The "PDF_ANNOTATIONS_HIGHLIGHT" callout is not fully visi        P1   0.8848       0.811         False
    2        Detect if a profile has Fx as default and then switched to Edge        P1   0.8841       0.737         False
    3 browser/components/attribution/test/browser/browser_AttributionCode_te        P1   0.8801       0.740         False
    4 [Experiment] Spotlight modal with Terms of Use information is no longe        P1   0.8700       0.795         False
    5               [META] [QA] [QA-1809] Feature name Easy Setup Onboarding        P1   0.8623       0.710         False
    6 Crash [@ js::gc::GetOrCreateUniqueId] or Assertion failure: env, at vm        P1   0.8621       0.724         False
    7        [META] [QA] [QA-1841] QA for FXE-182: Set to default for 1 week        P1   0.8600       0.698         False
    8 [Experiment] The “